# 02 Experiments (protocol v2): re-running the method described in the dissertation

Runs in Google Colab (GPU recommended). Implements dissertation Chapter 4 as the specification, with a clean protocol:

1. **Training pool** = DAIC-WOZ train + dev (140). **Held-out**: DAIC-WOZ official test (46) and E-DAIC (86 participants in no DAIC-WOZ split; its
   official test split of 56 is also scored as its own cohort), each scored once.
2. Architecture and graph hyperparameters fixed at the values the dissertation reports (Table 9). Optuna (TPE) tunes
   learning rate, weight decay, epochs and the class-weight multiplier m, by repeated stratified 5-fold CV inside the pool.
3. **Cross-fitted ensemble**: 5 folds x 4 seeds = 20 members. The threshold is calibrated on the out-of-fold probabilities
   of these same members (fewest false negatives with false-positive rate <= 6/21).

v1 (results in `results/experiments/`) tuned 9 hyperparameters on 107 participants and calibrated the threshold on
different models than the ones that scored the cohorts; its operating points were unstable. v2 results go to
`results/experiments/v2/`.

Deliberate changes from the legacy code: paralinguistic tags counted correctly, no 0.5 s turn filter, tags removed from the
lexical view, psycholinguistic view from `data/mapping/phq8_dsm5_empath.json`.
A controlled transcription analysis scores the DAIC-WOZ test participants on their E-DAIC ASR transcripts;
it is written to separate files (`controlled_*`) and is never part of the held-out evaluation.
Configuration frozen in `results/paper1/FROZEN.md` before scoring the E-DAIC cohorts.
Only aggregates are written to the repository. Per-participant outputs go to `OUTPUT_DIR`, outside it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, sys
REPO = '/content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn'
os.environ['DAIC_DATA_DIR'] = '/content/drive/MyDrive/Colab Notebooks/datasets'
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/outputs/paper1/v2'   # outside the repository
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, REPO)

In [3]:
!pip install -q -r requirements.txt
!python -m pytest -q tests

................................                                         [100%]
=============================== warnings summary ===============================
tests/test_experiment.py::test_run_final[inductgcn-words-views0]
  /content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn/src/paper1_graph_model/models.py:53: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
    a = torch.sparse_coo_tensor(edge_index, edge_weight, (x.shape[0], x.shape[0]))

tests/test_experiment.py::test_run_final[gcn_gat-words-views2]
tests/test_experiment.py::test_run_final[gcn_gat-words-views2]
tests/test_experiment.py::test_r

## Configuration

In [4]:
import torch
N_TRIALS = 30          # Optuna trials per model (restricted space: 4 hyperparameters)
N_FOLDS = 5
N_REPEATS = 2          # repeated CV inside the objective
SEEDS_PER_FOLD = 4     # ensemble = N_FOLDS * SEEDS_PER_FOLD = 20 members
SEED = 42
MAX_FPR = 6 / 21
SPACE = 'restricted'   # or 'full' for all of dissertation Table 9
STUDY_VERSION = 'v2.1'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STORAGE = f'sqlite:///{OUTPUT_DIR}/optuna.db'   # resumable if the runtime disconnects
print(DEVICE, torch.__version__)

cuda 2.14.1+cu130


## Cohorts and features

In [5]:
from src.common import data
from src.common.features import EmpathScorer
from src.paper1_graph_model.corpus import build_cohorts
from src.paper1_graph_model.experiment import merge_cohorts

scorer = EmpathScorer()
cohorts = build_cohorts(data.data_root(), scorer, psycholinguistic='dsm5')
pool = merge_cohorts('daic_pool', cohorts['daic_train'], cohorts['daic_dev'])
held_out = [cohorts['daic_test'], cohorts['edaic'], cohorts['edaic_test']]
controlled = cohorts['asr_daic_test']   # controlled transcription analysis: never in held_out
for c in [pool] + held_out + [controlled]:
    print(f'{c.name:12s} n={len(c.y):4d} positives={int(c.y.sum()):3d}')

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


daic_pool    n= 140 positives= 42
daic_test    n=  46 positives= 14
edaic        n= 112 positives= 34


## Models (rows of dissertation Table 11)

In [6]:
from src.paper1_graph_model.experiment import ModelSpec
ALL = ('temporal', 'paralinguistic', 'psycholinguistic')
CATS = scorer.categories
SPECS = [
    ModelSpec('InductGCN',                      'inductgcn', 'words',  ()),
    ModelSpec('GAT text-only',                  'gat',       'words',  ()),
    ModelSpec('GCN+GAT hybrid',                 'gcn_gat',   'words',  ALL),
    ModelSpec('EmpathGCN',                      'gcn',       'empath', ('psycholinguistic',), CATS),
    ModelSpec('GAT-GCN-FeExtr (Empath graph)',  'gat',       'empath', ALL, CATS),
    ModelSpec('GAT-GCN-FeExtr (word graph)',    'gat',       'words',  ALL),
]

## Hyperparameter search (training pool only)

In [7]:
import json, pathlib
from src.paper1_graph_model.experiment import tune, hp_from_params
RESULTS = pathlib.Path('results/experiments/v2'); RESULTS.mkdir(parents=True, exist_ok=True)

best = {}
for spec in SPECS:
    study = tune(spec, pool, N_TRIALS, N_FOLDS, N_REPEATS, SEED, DEVICE, storage=STORAGE,
                 space=SPACE, study_name=f'{spec.name} [{STUDY_VERSION}]')
    best[spec.name] = {'params': study.best_params, 'cv_pr_auc': study.best_value,
                       'n_trials': len(study.trials), 'space': SPACE}
    print(f'{spec.name:32s} CV PR-AUC={study.best_value:.3f}  {study.best_params}')
(RESULTS / 'best_params.json').write_text(json.dumps(best, indent=2))

[I 2026-11-23 21:01:32,435] A new study created in RDB with name: InductGCN [v2.1]
/content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn/src/paper1_graph_model/models.py:53: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  a = torch.sparse_coo_tensor(edge_index, edge_weight, (x.shape[0], x.shape[0]))
[I 2026-11-23 21:01:52,356] Trial 0 finished with value: 0.47483264567672756 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.47483264567672756.
[I 2026-11-23 21:02:00,978] Trial 1 finished with value:

InductGCN                        CV PR-AUC=0.519  {'lr': 0.0004059611610484307, 'weight_decay': 0.0005603803105930283, 'epochs': 450, 'm': 2.8736874205941256}


[I 2026-11-23 21:08:51,321] Trial 0 finished with value: 0.4476358557314786 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.4476358557314786.
[I 2026-11-23 21:09:13,671] Trial 1 finished with value: 0.47388013699370457 and parameters: {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}. Best is trial 1 with value: 0.47388013699370457.
[I 2026-11-23 21:09:35,972] Trial 2 finished with value: 0.4396450768913198 and parameters: {'lr': 0.0015930522616241021, 'weight_decay': 0.0013035123791853842, 'epochs': 300, 'm': 4.909729556485983}. Best is trial 1 with value: 0.47388013699370457.
[I 2026-11-23 21:10:02,046] Trial 3 finished with value: 0.43809284653133873 and parameters: {'lr': 0.004622589001020831, 'weight_decay': 0.0001329377199163636, 'epochs': 350, 'm': 2.5502135295603017}. Best is trial 1 with value: 0.47388013699370457.

GAT text-only                    CV PR-AUC=0.474  {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}


[I 2026-11-23 21:24:16,522] Trial 0 finished with value: 0.4298617147627467 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.4298617147627467.
[I 2026-11-23 21:24:38,688] Trial 1 finished with value: 0.47936870040948715 and parameters: {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}. Best is trial 1 with value: 0.47936870040948715.
[I 2026-11-23 21:25:00,986] Trial 2 finished with value: 0.43512824037927955 and parameters: {'lr': 0.0015930522616241021, 'weight_decay': 0.0013035123791853842, 'epochs': 300, 'm': 4.909729556485983}. Best is trial 1 with value: 0.47936870040948715.
[I 2026-11-23 21:25:26,941] Trial 3 finished with value: 0.44365974503277755 and parameters: {'lr': 0.004622589001020831, 'weight_decay': 0.0001329377199163636, 'epochs': 350, 'm': 2.5502135295603017}. Best is trial 1 with value: 0.47936870040948715

GCN+GAT hybrid                   CV PR-AUC=0.493  {'lr': 0.00020057247980860625, 'weight_decay': 0.0038151909356390163, 'epochs': 450, 'm': 4.842623593702834}


... (3 outputs left out)

GAT-GCN-FeExtr (Empath graph)    CV PR-AUC=0.582  {'lr': 0.002411713410724161, 'weight_decay': 0.00011709584632677918, 'epochs': 600, 'm': 3.288036619899568}


[I 2026-11-23 22:15:04,542] Trial 0 finished with value: 0.44002243152659426 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.44002243152659426.
[I 2026-11-23 22:15:29,271] Trial 1 finished with value: 0.487867007613253 and parameters: {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}. Best is trial 1 with value: 0.487867007613253.
[I 2026-11-23 22:15:53,994] Trial 2 finished with value: 0.4341222067779281 and parameters: {'lr': 0.0015930522616241021, 'weight_decay': 0.0013035123791853842, 'epochs': 300, 'm': 4.909729556485983}. Best is trial 1 with value: 0.487867007613253.
[I 2026-11-23 22:16:22,841] Trial 3 finished with value: 0.4628397965142969 and parameters: {'lr': 0.004622589001020831, 'weight_decay': 0.0001329377199163636, 'epochs': 350, 'm': 2.5502135295603017}. Best is trial 1 with value: 0.487867007613253.
[I 202

GAT-GCN-FeExtr (word graph)      CV PR-AUC=0.498  {'lr': 0.0001794662726839833, 'weight_decay': 0.000981709307142642, 'epochs': 350, 'm': 3.0187305646600144}


1602

## Cross-fitted ensembles, scored once on each held-out cohort

In [8]:
import numpy as np, pandas as pd
from src.paper1_graph_model.experiment import run_final

def row(model, cohort, m):
    lo, hi = m['f1_macro_ci']
    return {'model': model, 'cohort': cohort, 'n': m['n'], 'positives': m['positives'],
            'tau': round(m['threshold'], 2), 'F1 macro': round(m['f1_macro'], 4),
            'F1 95% CI': f'[{lo:.3f}, {hi:.3f}]', 'recall': round(m['recall'], 3),
            'precision': round(m['precision'], 3), 'specificity': round(m['specificity'], 3),
            'ROC-AUC': round(m.get('roc_auc', np.nan), 3), 'PR-AUC': round(m.get('pr_auc', np.nan), 3),
            'FN': m['fn'], 'FP': m['fp']}

def rows_of(name, res):
    out = [row(name, 'pool (out-of-fold)', res['oof_metrics'])]
    return out + [row(name, c, m) for c, m in res['metrics'].items()]

results, rows = {}, []
for spec in SPECS:
    hp = hp_from_params(spec, best[spec.name]['params'])
    res = run_final(spec, hp, pool, held_out, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE)
    results[spec.name] = res
    rows += rows_of(spec.name, res)
    print(spec.name, 'tau =', res['tau'], res['tau_info'])
metrics = pd.DataFrame(rows)
metrics.to_csv(RESULTS / 'metrics.csv', index=False)
metrics

InductGCN tau = 0.6 {'fn': 18, 'fp': 30, 'feasible': False, 'max_fpr': 0.2857142857142857}
GAT text-only tau = 0.6 {'fn': 18, 'fp': 30, 'feasible': False, 'max_fpr': 0.2857142857142857}
GCN+GAT hybrid tau = 0.6 {'fn': 17, 'fp': 34, 'feasible': False, 'max_fpr': 0.2857142857142857}
EmpathGCN tau = 0.42 {'fn': 18, 'fp': 28, 'feasible': True, 'max_fpr': 0.2857142857142857}
GAT-GCN-FeExtr (Empath graph) tau = 0.53 {'fn': 20, 'fp': 23, 'feasible': True, 'max_fpr': 0.2857142857142857}
GAT-GCN-FeExtr (word graph) tau = 0.6 {'fn': 18, 'fp': 28, 'feasible': True, 'max_fpr': 0.2857142857142857}


,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,InductGCN,pool (out-of-fold),140,42,0.60,0.6196,"[0.533, 0.698]",0.571,0.444,0.694,0.676,0.532,18,30
1,InductGCN,daic_test,46,14,0.60,0.6734,"[0.518, 0.802]",0.714,0.500,0.688,0.650,0.482,4,10
2,InductGCN,edaic,112,34,0.60,0.7130,"[0.618, 0.797]",0.706,0.558,0.756,0.758,0.652,10,19
3,GAT text-only,pool (out-of-fold),140,42,0.60,0.6196,"[0.536, 0.695]",0.571,0.444,0.694,0.680,0.537,18,30
4,GAT text-only,daic_test,46,14,0.60,0.5984,"[0.450, 0.732]",0.571,0.421,0.656,0.636,0.460,6,11
5,GAT text-only,edaic,112,34,0.60,0.6814,"[0.591, 0.764]",0.647,0.524,0.744,0.744,0.646,12,20
6,GCN+GAT hybrid,pool (out-of-fold),140,42,0.60,0.6051,"[0.524, 0.681]",0.595,0.424,0.653,0.674,0.556,17,34
7,GCN+GAT hybrid,daic_test,46,14,0.60,0.6349,"[0.480, 0.760]",0.714,0.455,0.625,0.658,0.479,4,12
8,GCN+GAT hybrid,edaic,112,34,0.60,0.6773,"[0.582, 0.761]",0.676,0.511,0.718,0.765,0.633,11,22
9,EmpathGCN,pool (out-of-fold),140,42,0.42,0.6317,"[0.547, 0.716]",0.571,0.462,0.714,0.665,0.505,18,28


In [9]:
# Per-participant probabilities stay outside the repository.
for name, res in results.items():
    for c in held_out:
        out = pd.DataFrame({'Participant_ID': c.ids, 'label': c.y, 'prob': res['probs'][c.name]})
        out.to_csv(f"{OUTPUT_DIR}/probs_{name.replace(' ', '_')}_{c.name}.csv", index=False)

In [ ]:
# Controlled transcription analysis, kept apart from the held-out evaluation (separate files and keys):
# the DAIC-WOZ test participants scored on their E-DAIC ASR transcripts by the same ensembles and thresholds.
from src.paper1_graph_model.experiment import score_cohort
ctrl_rows = []
for spec in SPECS:
    sc = score_cohort(results[spec.name], spec, controlled, DEVICE)
    ctrl_rows.append(row(spec.name, controlled.name, sc['metrics']))
    pd.DataFrame({'Participant_ID': controlled.ids, 'label': controlled.y, 'prob': sc['probs']}).to_csv(
        f"{OUTPUT_DIR}/controlled_probs_{spec.name.replace(' ', '_')}_{controlled.name}.csv", index=False)
controlled_metrics = pd.DataFrame(ctrl_rows)
controlled_metrics.to_csv(RESULTS / 'controlled_asr_daic_test.csv', index=False)
controlled_metrics

## Ablation of the professional-feature views (main model, hyperparameters fixed)

In [ ]:
from dataclasses import replace
MAIN = 'GAT-GCN-FeExtr (Empath graph)'   # change after looking at the table above if needed
main = next(s for s in SPECS if s.name == MAIN)
hp_main = hp_from_params(main, best[main.name]['params'])
ablation = []
for drop in ALL + (None,):
    views = tuple(v for v in ALL if v != drop)
    spec = replace(main, name=f'without {drop}' if drop else 'all views', views=views)
    ablation += rows_of(spec.name, run_final(spec, hp_main, pool, held_out, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE))
ablation = pd.DataFrame(ablation)
ablation.to_csv(RESULTS / 'ablation_views.csv', index=False)
ablation

,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,without temporal,pool (out-of-fold),140,42,0.53,0.6011,"[0.518, 0.681]",0.500,0.429,0.714,0.666,0.499,21,28
1,without temporal,daic_test,46,14,0.53,0.6057,"[0.456, 0.739]",0.857,0.429,0.500,0.676,0.397,2,16
2,without temporal,edaic,112,34,0.53,0.6952,"[0.600, 0.780]",0.765,0.520,0.692,0.808,0.585,8,24
3,without paralinguistic,pool (out-of-fold),140,42,0.55,0.6216,"[0.536, 0.702]",0.548,0.451,0.714,0.642,0.489,19,28
4,without paralinguistic,daic_test,46,14,0.55,0.7310,"[0.598, 0.847]",0.929,0.542,0.656,0.799,0.533,1,11
5,without paralinguistic,edaic,112,34,0.55,0.6725,"[0.576, 0.761]",0.588,0.526,0.769,0.801,0.600,14,18
6,without psycholinguistic,pool (out-of-fold),140,42,0.60,0.6268,"[0.534, 0.708]",0.429,0.500,0.816,0.631,0.530,24,18
7,without psycholinguistic,daic_test,46,14,0.60,0.7433,"[0.586, 0.863]",0.643,0.643,0.844,0.839,0.609,5,5
8,without psycholinguistic,edaic,112,34,0.60,0.5827,"[0.478, 0.688]",0.206,0.778,0.974,0.796,0.619,27,2
9,all views,pool (out-of-fold),140,42,0.52,0.6459,"[0.554, 0.735]",0.548,0.489,0.755,0.660,0.537,19,24


## DSM-5 mapping vs. legacy Empath domain groups (EmpathGCN and main model)

Rebuilds the features with the legacy `EMPATH_DOMAIN_GROUPS` and repeats the final runs with the same hyperparameters.

In [ ]:
cohorts_dom = build_cohorts(data.data_root(), scorer, psycholinguistic='domains')
pool_dom = merge_cohorts('daic_pool', cohorts_dom['daic_train'], cohorts_dom['daic_dev'])
held_dom = [cohorts_dom['daic_test'], cohorts_dom['edaic']]
cmp_rows = []
for name in ['EmpathGCN', MAIN]:
    spec = next(s for s in SPECS if s.name == name)
    hp = hp_from_params(spec, best[name]['params'])
    res_dom = run_final(spec, hp, pool_dom, held_dom, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE)
    cmp_rows += rows_of(f'{name} / DSM-5 mapping', results[name]) + rows_of(f'{name} / legacy domains', res_dom)
mapping_vs_domains = pd.DataFrame(cmp_rows)
mapping_vs_domains.to_csv(RESULTS / 'mapping_vs_domains.csv', index=False)
mapping_vs_domains

,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,EmpathGCN / DSM-5 mapping,pool (out-of-fold),140,42,0.42,0.6317,"[0.547, 0.716]",0.571,0.462,0.714,0.665,0.505,18,28
1,EmpathGCN / DSM-5 mapping,daic_test,46,14,0.42,0.5965,"[0.440, 0.732]",0.714,0.417,0.562,0.699,0.473,4,14
2,EmpathGCN / DSM-5 mapping,edaic,112,34,0.42,0.7275,"[0.634, 0.809]",0.853,0.547,0.692,0.783,0.577,5,24
3,EmpathGCN / legacy domains,pool (out-of-fold),140,42,0.58,0.6438,"[0.556, 0.727]",0.571,0.480,0.735,0.672,0.526,18,26
4,EmpathGCN / legacy domains,daic_test,46,14,0.58,0.5687,"[0.409, 0.706]",0.500,0.389,0.656,0.690,0.458,7,11
5,EmpathGCN / legacy domains,edaic,112,34,0.58,0.6492,"[0.555, 0.743]",0.588,0.488,0.731,0.768,0.565,14,21
6,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,pool (out-of-fold),140,42,0.53,0.6415,"[0.552, 0.727]",0.524,0.489,0.765,0.657,0.536,20,23
7,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,daic_test,46,14,0.53,0.7464,"[0.616, 0.867]",0.857,0.571,0.719,0.812,0.562,2,9
8,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,edaic,112,34,0.53,0.6959,"[0.595, 0.786]",0.500,0.630,0.872,0.810,0.608,17,10
9,GAT-GCN-FeExtr (Empath graph) / legacy domains,pool (out-of-fold),140,42,0.55,0.6459,"[0.558, 0.724]",0.548,0.489,0.755,0.658,0.541,19,24


## Attention over Empath categories (auditability)

Mean attention from each category node to each test participant, averaged over heads and the 20 members.
Per-participant matrices go to `OUTPUT_DIR`; only the class-level ranking is written to the repository.

In [ ]:
from src.paper1_graph_model.experiment import attention_to_vocabulary
test_c = cohorts['daic_test']
att = attention_to_vocabulary(results[main.name], main, test_c, DEVICE)
att.to_csv(f"{OUTPUT_DIR}/attention_test_{main.name.replace(' ', '_')}.csv")
by_class = pd.DataFrame({'control': att[test_c.y == 0].mean(), 'depressed': att[test_c.y == 1].mean()})
by_class['delta'] = by_class['depressed'] - by_class['control']
by_class = by_class.sort_values('delta', ascending=False)
by_class.to_csv(RESULTS / 'attention_by_class_test.csv')
by_class.head(20)

,control,depressed,delta
neglect,0.005193,0.009993,0.004799
furniture,0.002699,0.006547,0.003848
morning,0.007187,0.010819,0.003632
health,0.007147,0.010656,0.003508
gain,0.004613,0.007830,0.003217
masculine,0.007000,0.009967,0.002967
timidity,0.009581,0.012539,0.002958
aggression,0.007105,0.009932,0.002827
royalty,0.003138,0.005482,0.002344
real_estate,0.006884,0.009213,0.002329
